In [1]:
import pandas as pd

# load all stair-relevant attribute train files
train_files = {
    "fall_height": pd.read_csv("../data/processed/train/attr_fall_height_train.csv"),
    "has_pedestrian_railing": pd.read_csv("../data/processed/train/attr_has_pedestrian_railing_train.csv"),
    "material": pd.read_csv("../data/processed/train/attr_material_frame_tank_body_train.csv"),
    "number_of_steps": pd.read_csv("../data/processed/train/attr_number_of_steps_train.csv"),
    "structure_position": pd.read_csv("../data/processed/train/attr_structure_position_train.csv"),
}

# filter each to stairs only
stair_assets = {}
for attr, df in train_files.items():
    stair_assets[attr] = set(
        df[df["profile_name"] == "Stairs"]["asset_id"].unique()
    )

# find overlap across all 5
overlap = set.intersection(*stair_assets.values())
print(f"Assets in all 5 stair attribute train files: {len(overlap)}")
print(list(overlap)[:5])

Assets in all 5 stair attribute train files: 3
[np.int64(126777), np.int64(121499), np.int64(121503)]


In [2]:
test_asset_id = 126777

# verify it exists in all files
for attr, df in train_files.items():
    row = df[df["asset_id"] == test_asset_id]
    print(f"{attr}: {row[f'attr_{attr}' if attr != 'material' else 'attr_material_frame,_tank,_body'].values}")

fall_height: [0.5]
has_pedestrian_railing: ['1 railing']
material: ['Timber/Wood']
number_of_steps: [12.]
structure_position: ['Elevated']


In [3]:
# use structure position train file since it has all columns including image_path
test_df = train_files["structure_position"][
    train_files["structure_position"]["asset_id"] == test_asset_id
]

print(f"Images for asset {test_asset_id}: {len(test_df)}")
test_df[["asset_id", "profile_name", "image_path"]].to_csv(
    "../data/processed/train/pipeline_test_asset.csv", index=False
)

Images for asset 126777: 1


In [ ]:
#comparing test with ground truth labels

train_files = {
    "attr_fall_height": pd.read_csv("../data/processed/train/attr_fall_height_train.csv"),
    "attr_has_pedestrian_railing": pd.read_csv("../data/processed/train/attr_has_pedestrian_railing_train.csv"),
    "attr_material_frame,_tank,_body": pd.read_csv("../data/processed/train/attr_material_frame_tank_body_train.csv"),
    "attr_number_of_steps": pd.read_csv("../data/processed/train/attr_number_of_steps_train.csv"),
    "attr_structure_position": pd.read_csv("../data/processed/train/attr_structure_position_train.csv"),
}

for attr, df in train_files.items():
    row = df[df["asset_id"] == 126777]
    if len(row) > 0:
        print(f"{attr}: {row[attr].values[0]}")

attr_fall_height: 0.5
attr_has_pedestrian_railing: 1 railing
attr_material_frame,_tank,_body: Timber/Wood
attr_number_of_steps: 12.0
attr_structure_position: Elevated


In [6]:
PROMPT_TEMPLATE = """
    You are an expert in park infrastructure analysis.

    Using ALL provided images of this single stair asset, identify the most likely
    attribute values. For each of the following attributes, the possible values are
    given below. Predict exactly ONE value from the listed options for each
    attribute, and provide a confidence score (0.0-1.0) for each prediction.

    Attributes to predict:
    - fall_height: low (<0.5m) | medium (0.5m-1.2m) | high (>1.2m)
    - has_pedestrian_railing: 2 railings | 1 railing | no railings
    - material_frame_tank_body: PVC | Gravel | Natural Surface | Earth-filled |
                                Aluminum | Metal | Steel | Rock/Stone | Concrete |
                                Box Step | Timber/Wood
    - number_of_steps: few (<10) | medium (10-20) | many (>20)
    - structure_position: Elevated | At-Grade | Other

    Return ONLY a valid JSON object with this exact schema (no markdown, no prose):
    {
        "<attribute_key>": {
        "value": "<predicted value or 'unable to determine'>",
        "confidence": <float 0.0-1.0>
        }
    }

    If you cannot determine an attribute from the images, set value to
    "unable to determine" and confidence to 0.0.
    """

In [9]:
import sys
from pathlib import Path

# add project root to path
sys.path.insert(0, str(Path("..").resolve()))

from src.vlm.predictors import predict_asset_attributes
import pandas as pd

df = pd.read_csv("../data/processed/train/pipeline_test_asset.csv")

result = predict_asset_attributes(
    asset_id=126777,
    df=df,
    model_name="gemini-3-flash-preview",
    prompt=PROMPT_TEMPLATE
)

print(result)

{'asset_id': 126777, 'error': "503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}", 'response': None}
